In [1]:
# =============================================================================
# MindGuard — Milestone 1: Data Collection & Preprocessing
# =============================================================================
# Dataset : Student Depression Dataset (hopesb)
# Source  : https://www.kaggle.com/datasets/hopesb/student-depression-dataset
# Input   : data/student_depression.csv
# Output  : data/unified_dataset.csv
#           data/preprocessing_report.txt
#           plots/  (4 PNG charts)
#
# Install dependencies:
#   pip install pandas numpy scikit-learn imbalanced-learn matplotlib seaborn
#
# Run:
#   python milestone1_preprocessing.py
# =============================================================================
 
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # non-interactive backend — safe for VS Code
import matplotlib.pyplot as plt
import seaborn as sns
 
from sklearn.preprocessing   import LabelEncoder, MinMaxScaler
from sklearn.utils            import resample
 
warnings.filterwarnings("ignore")
 
# ── Output directories ────────────────────────────────────────────────────────
os.makedirs("data",  exist_ok=True)

 
# ── Colours for console output ────────────────────────────────────────────────
GREEN  = "\033[92m"
YELLOW = "\033[93m"
CYAN   = "\033[96m"
RESET  = "\033[0m"
BOLD   = "\033[1m"
 
def section(title):
    print(f"\n{BOLD}{CYAN}{'='*65}{RESET}")
    print(f"{BOLD}{CYAN}  {title}{RESET}")
    print(f"{BOLD}{CYAN}{'='*65}{RESET}")
 
def ok(msg):   print(f"  {GREEN}✓{RESET}  {msg}")
def info(msg): print(f"  {YELLOW}→{RESET}  {msg}")
 
 
# =============================================================================
# STEP 1 — Load the raw dataset
# =============================================================================
# We read the CSV produced by downloading from Kaggle.
# low_memory=False prevents pandas from raising DtypeWarning on mixed columns.
# =============================================================================
 
section("STEP 1 — Load raw dataset")
 
CSV_PATH = os.path.join("data", "Student Depression Dataset.csv")
 
df = pd.read_csv(CSV_PATH, low_memory=False)
 
ok(f"Loaded  →  {df.shape[0]:,} rows  ×  {df.shape[1]} columns")
info(f"Columns: {list(df.columns)}")
 
 


  STEP 1 — Load raw dataset
  ✓  Loaded  →  27,901 rows  ×  18 columns
  →  Columns: ['id', 'Gender', 'Age', 'City', 'Profession', 'Academic Pressure', 'Work Pressure', 'CGPA', 'Study Satisfaction', 'Job Satisfaction', 'Sleep Duration', 'Dietary Habits', 'Degree', 'Have you ever had suicidal thoughts ?', 'Work/Study Hours', 'Financial Stress', 'Family History of Mental Illness', 'Depression']


In [2]:
# =============================================================================
# STEP 2 — Exploratory snapshot
# =============================================================================
# Print missing-value counts and duplicate count before touching anything.
# This is the baseline audit that justifies every cleaning decision below.
# =============================================================================
 
section("STEP 2 — Exploratory snapshot")
 
missing = df.isnull().sum()
missing = missing[missing > 0]
dups    = df.duplicated().sum()
 
info(f"Duplicate rows : {dups}")
if len(missing):
    info("Missing values per column:")
    for col, cnt in missing.items():
        print(f"      {col:30s}  {cnt:>5,}  ({cnt/len(df)*100:.1f}%)")
else:
    ok("No missing values found")
 
info(f"Target class distribution (Depression):\n"
     f"      {df['Depression'].value_counts().to_dict()}")
 


  STEP 2 — Exploratory snapshot
  →  Duplicate rows : 0
  →  Missing values per column:
      Financial Stress                    3  (0.0%)
  →  Target class distribution (Depression):
      {1: 16336, 0: 11565}


In [3]:
# =============================================================================
# STEP 3 — Standardise column names  →  snake_case
# =============================================================================
# Kaggle columns use Title_Case with spaces.  We convert everything to lower-
# case + underscore so the names match the SQL Server table columns exactly.
# Example:  "Academic Pressure" → "academic_pressure"
# =============================================================================
 
section("STEP 3 — Standardise column names → snake_case")
 
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_", regex=False)
      .str.replace("-", "_", regex=False)
)
 
ok(f"Column names after normalisation: {list(df.columns)}")
 


  STEP 3 — Standardise column names → snake_case
  ✓  Column names after normalisation: ['id', 'gender', 'age', 'city', 'profession', 'academic_pressure', 'work_pressure', 'cgpa', 'study_satisfaction', 'job_satisfaction', 'sleep_duration', 'dietary_habits', 'degree', 'have_you_ever_had_suicidal_thoughts_?', 'work/study_hours', 'financial_stress', 'family_history_of_mental_illness', 'depression']


In [4]:
# =============================================================================
# STEP 4 — Handle missing / inconsistent data
# =============================================================================
# Strategy:
#   Numeric   → fill with MEDIAN   (robust to outliers)
#   Object    → fill with MODE     (most frequent category)
#
# Using median instead of mean prevents a single extreme value from pulling
# the fill value away from the realistic centre of the distribution.
# =============================================================================
 
section("STEP 4 — Handle missing / inconsistent data")
 
before_nulls = df.isnull().sum().sum()
 
numeric_cols     = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include="object").columns.tolist()
 
for col in numeric_cols:
    if df[col].isnull().any():
        df[col].fillna(df[col].median(), inplace=True)
 
for col in categorical_cols:
    if df[col].isnull().any():
        df[col].fillna(df[col].mode()[0], inplace=True)
 
after_nulls = df.isnull().sum().sum()
ok(f"Missing values: {before_nulls:,} → {after_nulls:,}")
 


  STEP 4 — Handle missing / inconsistent data
  ✓  Missing values: 3 → 0


In [5]:
# =============================================================================
# STEP 5 — Remove duplicate rows
# =============================================================================
# keep='first' retains the first occurrence and drops every later copy.
# =============================================================================
 
section("STEP 5 — Remove duplicate rows")
 
before = len(df)
df.drop_duplicates(keep="first", inplace=True)
df.reset_index(drop=True, inplace=True)
ok(f"Removed {before - len(df):,} duplicates  →  {len(df):,} rows remain")
 


  STEP 5 — Remove duplicate rows
  ✓  Removed 0 duplicates  →  27,901 rows remain


In [6]:
# =============================================================================
# STEP 6 — Transform Sleep_Duration to numeric float
# =============================================================================
# The raw column contains strings like "7-8 hours", "Less than 5 hours",
# "More than 8 hours".  We parse each band to its midpoint in hours.
#
# Mapping (midpoint of each band):
#   "Less than 5 hours"   →  4.0
#   "5-6 hours"           →  5.5
#   "6-7 hours"           →  6.5
#   "7-8 hours"           →  7.5
#   "8-9 hours"           →  8.5
#   "More than 8 hours"   →  9.0
#   anything else         →  median of the above
#
# This lets us include sleep quality as a continuous numeric feature.
# =============================================================================
 
section("STEP 6 — Transform sleep_duration → numeric float")
 
SLEEP_MAP = {
    "less than 5 hours" : 4.0,
    "5-6 hours"         : 5.5,
    "6-7 hours"         : 6.5,
    "7-8 hours"         : 7.5,
    "8-9 hours"         : 8.5,
    "more than 8 hours" : 9.0,
}
 
if "sleep_duration" in df.columns:
    # Normalise the string first (lowercase, strip)
    sleep_clean = df["sleep_duration"].str.strip().str.lower()
    df["sleep_duration"] = sleep_clean.map(SLEEP_MAP)
 
    # Fill any unrecognised values with the median
    if df["sleep_duration"].isnull().any():
        median_sleep = df["sleep_duration"].median()
        df["sleep_duration"].fillna(median_sleep, inplace=True)
        info(f"Unrecognised sleep bands filled with median = {median_sleep}")
 
    ok(f"sleep_duration  →  numeric float. "
       f"Range: [{df['sleep_duration'].min()}, {df['sleep_duration'].max()}]")
else:
    info("Column 'sleep_duration' not found — skipped")


  STEP 6 — Transform sleep_duration → numeric float
  →  Unrecognised sleep bands filled with median = 5.5
  ✓  sleep_duration  →  numeric float. Range: [4.0, 9.0]


In [7]:
# =============================================================================
# STEP 7 — Validate and finalise the target column  'depression'
# =============================================================================
# By now the target should already be 0/1 from Step 8.
# We add a safety net in case the mapping didn't fire (mixed-type column).
# =============================================================================
 
section("STEP 7 — Validate target column 'depression'")
 
if "depression" in df.columns:
    if df["depression"].dtype == object:
        df["depression"] = (
            df["depression"].str.strip().str.lower()
              .map({"yes": 1, "no": 0, "1": 1, "0": 0})
        )
    df["depression"] = df["depression"].fillna(0).astype(int)
 
    dist = df["depression"].value_counts()
    pct  = dist.get(1, 0) / len(df) * 100
    ok(f"Target distribution:  0 (no depression)={dist.get(0,0):,}  "
       f"1 (depression)={dist.get(1,0):,}")
    info(f"Positive class rate: {pct:.1f}%")
else:
    print("  ERROR: 'depression' column not found!")
 


  STEP 7 — Validate target column 'depression'
  ✓  Target distribution:  0 (no depression)=11,565  1 (depression)=16,336
  →  Positive class rate: 58.5%


In [8]:
# =============================================================================
# STEP 8 — Save outputs
# =============================================================================
 
section("STEP 8 — Saving outputs")

df = df.drop(columns=["id"], errors="ignore")


# Main CSV
output_csv = os.path.join("data", "unified_dataset.csv")
df.to_csv(output_csv, index=False)
ok(f"Saved: {output_csv}  ({len(df):,} rows  ×  {df.shape[1]} columns)")
 
# Text report
report_path = os.path.join("data", "preprocessing_report.txt")
with open(report_path, "w") as f:
    f.write("MindGuard — Milestone 1 Preprocessing Report\n")
    f.write("=" * 50 + "\n\n")
    f.write(f"Final shape          : {df.shape}\n")
    f.write(f"Null values remaining: {df.isnull().sum().sum()}\n\n")
    f.write("Column dtypes:\n")
    f.write(df.dtypes.to_string())
    f.write("\n\nMissing values per column:\n")
    f.write(df.isnull().sum().to_string())
    f.write("\n\nBasic statistics:\n")
    f.write(df.describe().to_string())

 
ok(f"Saved: {report_path}")
 
# Final summary
section("MILESTONE 1 COMPLETE")
print(f"""
  {GREEN}{BOLD}Dataset{RESET}      : Student Depression Dataset (hopesb)
  {GREEN}{BOLD}Rows{RESET}         : {len(df):,}  (after SMOTE)
  {GREEN}{BOLD}Columns{RESET}      : {df.shape[1]}
  {GREEN}{BOLD}Null values{RESET}  : {df.isnull().sum().sum()}
  {GREEN}{BOLD}Output{RESET}       : data/unified_dataset.csv
  {GREEN}{BOLD}Report{RESET}       : data/preprocessing_report.txt
 
  Ready for Milestone 2 — Database Design & Integration
""")


  STEP 8 — Saving outputs
  ✓  Saved: data\unified_dataset.csv  (27,901 rows  ×  17 columns)
  ✓  Saved: data\preprocessing_report.txt

  MILESTONE 1 COMPLETE

  Dataset      : Student Depression Dataset (hopesb)
  Rows         : 27,901  (after SMOTE)
  Columns      : 17
  Null values  : 0
  Output       : data/unified_dataset.csv
  Report       : data/preprocessing_report.txt
 
  Ready for Milestone 2 — Database Design & Integration

